# CUDA SAXPY\n\nChoose **Runtime -> Change runtime type -> T4 GPU**, then run all cells.

In [ ]:
!nvidia-smi

In [ ]:
%%writefile saxpy.cu\n#include <math.h>\n#include <stdio.h>\n\n__global__ void saxpy(int n, double a, const double *x, double *y) {\n    int i = blockIdx.x * blockDim.x + threadIdx.x;\n    if (i < n) y[i] = a * x[i] + y[i];\n}\n\nint main(void) {\n    const long N = 1L << 24;\n    double *x, *y;\n    cudaMallocManaged(&x, N * sizeof(double));\n    cudaMallocManaged(&y, N * sizeof(double));\n\n    for (long i = 0; i < N; i++) { x[i] = 1.0; y[i] = 2.0; }\n\n    cudaEvent_t start, stop;\n    cudaEventCreate(&start);\n    cudaEventCreate(&stop);\n\n    int block = 256;\n    int grid = (N + block - 1) / block;\n    cudaEventRecord(start);\n    saxpy<<<grid, block>>>(N, 3.0, x, y);\n    cudaEventRecord(stop);\n    cudaEventSynchronize(stop);\n\n    float ms = 0;\n    cudaEventElapsedTime(&ms, start, stop);\n    double gbps = (3.0 * N * sizeof(double)) / (ms * 1e6);\n    printf("N=%ld, time=%.3f ms, %.1f GB/s\\n", N, ms, gbps);\n\n    double maxerr = 0;\n    for (long i = 0; i < N; i++) {\n        double err = fabs(y[i] - 5.0);\n        if (err > maxerr) maxerr = err;\n    }\n    printf("max error = %g\\n", maxerr);\n\n    cudaFree(x);\n    cudaFree(y);\n    return 0;\n}\n

In [ ]:
!nvcc -O3 saxpy.cu -o saxpy\n!./saxpy